Build Gold — Live Production Data
Reads silver battery data, builds weekly and monthly aggregated features, saves to gold.

**Input**: silver/erp/battery/battery_clean_live.json
**Output**: gold/erp/battery/phase1_overall_weekly_live.parquet, phase1_overall_monthly_live.parquet

In [0]:
%run ../../_local_config

In [0]:
%pip install rapidfuzz

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
import pandas as pd
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.transform.battery.build_gold_features import build_gold_overall_weekly, build_gold_overall_monthly

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery"

In [0]:
silver = read_silver(blob_service, "live/battery_clean_live.json")
silver["posting_date"] = pd.to_datetime(silver["posting_date"])

print(f"Silver: {silver.shape}")
print(f"Date range: {silver['posting_date'].min()} to {silver['posting_date'].max()}")

Sales prediction

In [0]:
# Weekly gold — safe to use all data
gold_weekly = build_gold_overall_weekly(silver)

# Monthly gold — trim to the last COMPLETE month first
monthly_cutoff = silver["posting_date"].max().replace(day=1) - pd.Timedelta(days=1)
silver_for_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()
gold_monthly = build_gold_overall_monthly(silver_for_monthly)

print(f"Weekly gold: {gold_weekly.shape}")
print(gold_weekly.tail(5)[["week_start", "total_units_sold"]])

print(f"\nMonthly gold (cutoff: {monthly_cutoff.date()}): {gold_monthly.shape}")
print(gold_monthly.tail(5)[["month_start", "total_units_sold"]])

In [0]:
print(f"Filled weeks: {gold_weekly['was_filled'].sum()} / {len(gold_weekly)}")
print(f"Filled months: {gold_monthly['was_filled'].sum()} / {len(gold_monthly)}")

In [0]:
save_gold(blob_service, gold_weekly, f"{FORECAST_BASE}/data/phase1_overall_weekly_live.parquet")
save_gold(blob_service, gold_monthly, f"{FORECAST_BASE}/data/phase1_overall_monthly_live.parquet")
print("Saved weekly and monthly gold")

Brand wise prediction

In [0]:
from src.transform.build_gold_features import (
    build_gold_brand_weekly, build_gold_brand_monthly
)

gold_brand_weekly = build_gold_brand_weekly(silver)

monthly_cutoff = silver["posting_date"].max().replace(day=1) - pd.Timedelta(days=1)
silver_for_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()
gold_brand_monthly = build_gold_brand_monthly(silver_for_monthly)

print(f"Brand weekly: {gold_brand_weekly.shape}")
print(gold_brand_weekly.groupby("brand_code")["total_units_sold"].sum())

print(f"\nBrand monthly: {gold_brand_monthly.shape}")
print(gold_brand_monthly.groupby("brand_code")["total_units_sold"].sum())

In [0]:
save_gold(blob_service, gold_brand_weekly,f"{FORECAST_BASE}/data/phase2_brand_weekly_live.parquet")
save_gold(blob_service, gold_brand_monthly, f"{FORECAST_BASE}/data/phase2_brand_monthly_live.parquet")
print("Saved brand weekly and monthly gold")

Vehicle prediction

In [0]:
from src.transform.build_gold_features import build_gold_vehicle_weekly, build_gold_vehicle_monthly

# Determine active vehicle types dynamically (recency + volume based, no hardcoding)
recency_threshold = silver["posting_date"].max() - pd.Timedelta(days=180)
vehicle_last_sale = silver.groupby("vehicle_type")["posting_date"].max().reset_index()
vehicle_last_sale.columns = ["vehicle_type", "last_sale_date"]
vehicle_last_sale["is_active"] = vehicle_last_sale["last_sale_date"] >= recency_threshold

vehicle_types_to_forecast = vehicle_last_sale[vehicle_last_sale["is_active"]]["vehicle_type"].tolist()
print(f"Active vehicle types: {vehicle_types_to_forecast}")

silver_vehicles = silver[silver["vehicle_type"].isin(vehicle_types_to_forecast)].copy()

gold_vehicle_weekly = build_gold_vehicle_weekly(silver_vehicles)
gold_vehicle_monthly = build_gold_vehicle_monthly(silver_vehicles[silver_vehicles["posting_date"] <= monthly_cutoff])

print(f"Vehicle weekly: {gold_vehicle_weekly.shape}")
print(f"Vehicle monthly: {gold_vehicle_monthly.shape}")

save_gold(blob_service, gold_vehicle_weekly, f"{FORECAST_BASE}/data/phase3_vehicle_weekly_live.parquet")
save_gold(blob_service, gold_vehicle_monthly, f"{FORECAST_BASE}/data/phase3_vehicle_monthly_live.parquet")
print("Saved vehicle weekly and monthly gold")

Location prediction

In [0]:
from src.transform.build_gold_features import build_gold_location_weekly, build_gold_location_monthly

# Determine active locations dynamically (recency + volume, no hardcoding)
location_stats = silver.groupby(["location_code", "location_description"]).agg(
    total_net_units=("net_units", "sum"),
    last_sale_date=("posting_date", "max")
).reset_index()

recency_threshold = silver["posting_date"].max() - pd.Timedelta(days=180)
location_stats["is_active"] = (
    (location_stats["last_sale_date"] >= recency_threshold) &
    (location_stats["total_net_units"] >= 1000)
)

locations_to_forecast = location_stats[location_stats["is_active"]]["location_code"].tolist()
print(f"Active locations: {locations_to_forecast}")

silver_locations = silver[silver["location_code"].isin(locations_to_forecast)].copy()

gold_location_weekly = build_gold_location_weekly(silver_locations)
gold_location_monthly = build_gold_location_monthly(silver_locations[silver_locations["posting_date"] <= monthly_cutoff])

print(f"Location weekly: {gold_location_weekly.shape}")
print(f"Location monthly: {gold_location_monthly.shape}")

save_gold(blob_service, gold_location_weekly, f"{FORECAST_BASE}/data/phase4_location_weekly_live.parquet")
save_gold(blob_service, gold_location_monthly, f"{FORECAST_BASE}/data/phase4_location_monthly_live.parquet")
print("Saved location weekly and monthly gold")

In [0]:
from rapidfuzz import process, fuzz
import io

# Load the city/district/province reference table built earlier
blob_client = blob_service.get_blob_client(container="silver", blob=f"{FORECAST_BASE}/reference/location_hierarchy.parquet")
stream = blob_client.download_blob().readall()
location_hierarchy = pd.read_parquet(io.BytesIO(stream))

city_names = location_hierarchy["city_name"].unique().tolist()
city_names_upper = [c.upper() for c in city_names]
upper_to_original = dict(zip(city_names_upper, city_names))

def find_city_fuzzy(description, threshold=85):
    words = description.upper().split()
    candidates = words + [" ".join(words[i:i+2]) for i in range(len(words) - 1)]
    best_match, best_score = None, 0
    for candidate in candidates:
        result = process.extractOne(candidate, city_names_upper, scorer=fuzz.ratio)
        if result and result[1] > best_score and result[1] >= threshold:
            best_match = upper_to_original[result[0]]
            best_score = result[1]
    return best_match if best_match else "Others"

location_reference = location_stats[location_stats["is_active"]][["location_code", "location_description"]].copy()
location_reference["matched_city"] = location_reference["location_description"].apply(find_city_fuzzy)

location_reference["match_key"] = location_reference["matched_city"].str.upper()
location_hierarchy["match_key"] = location_hierarchy["city_name"].str.upper()

location_reference = location_reference.merge(
    location_hierarchy[["match_key", "district_name", "province_name"]],
    on="match_key", how="left"
).drop(columns=["match_key"])

location_reference["district_name"] = location_reference["district_name"].fillna("Others")
location_reference["province_name"] = location_reference["province_name"].fillna("Others")

print(location_reference)

save_gold(blob_service, location_reference, f"{FORECAST_BASE}/data/reference/location_reference_live.parquet")
print("Saved location reference table")

In [0]:
item_stats = silver.groupby("item_no").agg(
    total_net_units=("net_units", "sum"),
    last_sale_date=("posting_date", "max")
).reset_index()

recency_threshold = silver["posting_date"].max() - pd.Timedelta(days=180)
item_stats["is_active"] = (
    (item_stats["last_sale_date"] >= recency_threshold) &
    (item_stats["total_net_units"] >= 1000)
)

active_items_df = item_stats[item_stats["is_active"]].sort_values("total_net_units", ascending=False)
print(f"Active items: {len(active_items_df)}")
print(active_items_df)

inactive_volume = item_stats[~item_stats["is_active"]]["total_net_units"].sum()
total_volume = item_stats["total_net_units"].sum()
print(f"\nExcluded volume: {inactive_volume:,.0f} ({inactive_volume/total_volume:.1%} of total)")

In [0]:
active_item_list = active_items_df["item_no"].tolist()

silver_items = silver.copy()
silver_items["item_group"] = silver_items["item_no"].where(
    silver_items["item_no"].isin(active_item_list), "OTHERS"
)

print(silver_items.groupby("item_group")["net_units"].sum().sort_values(ascending=False).head(10))
print(f"\nTotal item groups (including OTHERS): {silver_items['item_group'].nunique()}")

In [0]:
from src.transform.build_gold_features import build_gold_item_weekly, build_gold_item_monthly

gold_item_weekly = build_gold_item_weekly(silver_items)
gold_item_monthly = build_gold_item_monthly(silver_items[silver_items["posting_date"] <= monthly_cutoff])

print(f"Item weekly: {gold_item_weekly.shape}")
print(f"Item monthly: {gold_item_monthly.shape}")

save_gold(blob_service, gold_item_weekly, f"{FORECAST_BASE}/data/phase5_item_weekly_live.parquet")
save_gold(blob_service, gold_item_monthly, f"{FORECAST_BASE}/data/phase5_item_monthly_live.parquet")
print("Saved item weekly and monthly gold")